# **Unsupervised Burned Area Detection using Spectral Indices and Isolation Forest**

## Usage Instructions
1. Clone this repository to your local machine.
2. Create a folder named `data/` in the root directory.
3. Place your pre-processed `.tif` files inside the `data/` folder.
4. Launch Jupyter Notebook or JupyterLab from your terminal.
5. Open the `burned_area_detection_spectralindex_Isolation_forest.ipynb` notebook and run all cells sequentially.

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# 1. SPATIAL ALIGNMENT FUNCTION

In [ ]:

def reproject_raster(src_path, ref_profile):
    """
    Reprojects a source raster to match the spatial profile of a reference raster.
    This ensures all raster layers share the same CRS, resolution, and extent.
    """
    with rasterio.open(src_path) as src:
        # Extract source and destination geospatial metadata
        src_transform = src.transform
        src_crs = src.crs
        dst_transform = ref_profile['transform']
        dst_crs = ref_profile['crs']
        dst_shape = (ref_profile['height'], ref_profile['width'])

        # Initialize an empty array for the reprojected data
        dst_data = np.empty(dst_shape, dtype=src.read(1).dtype)

        # Perform the reprojection using bilinear resampling for continuous data (spectral indices)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_data,
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )

    return dst_data

# 2. DATA LOADING AND PREPROCESSING

In [ ]:
# File paths for the spectral index difference rasters (August - May)
file_paths = [
    r"data\NDVI_diff_august_may_2024.tif",
    r"data\NBR_diff_august_may_2024.tif",
    r"data\NDMI_diff_august_may_2024.tif"
]

# Load the reference profile from the first file to establish the baseline grid
with rasterio.open(file_paths[0]) as ref_src:
    ref_profile = ref_src.profile

# Reproject all rasters to the reference profile and store them in a list
raster_data = []
for file_path in file_paths:
    aligned_data = reproject_raster(file_path, ref_profile)
    raster_data.append(aligned_data)

# Create a 3D numpy array (height, width, features/bands)
raster_stack = np.stack(raster_data, axis=-1)
height, width, num_features = raster_stack.shape

# 3. NODATA MASKING (CRITICAL FIX)

In [ ]:
# Identify valid pixels. We assume NoData might be represented as NaNs.
# If your rasters use a specific NoData value (e.g., -9999), replace np.isnan with:
# raster_stack == -9999
valid_mask = ~np.isnan(raster_stack).any(axis=-1)

# Extract only the valid pixels for the machine learning model.
# This flattens the spatial dimensions into a 2D array: (N_valid_pixels, 3 features)
valid_pixels = raster_stack[valid_mask]


# 4. ANOMALY DETECTION (MACHINE LEARNING)


In [ ]:

# Initialize the Isolation Forest model.
# Contamination is set to 2.5%, representing the expected proportion of burned area.
model = IsolationForest(contamination=0.025, random_state=42)

# Fit the model exclusively on valid pixels
model.fit(valid_pixels)

# CRITICAL FIX: Use decision_function to get continuous anomaly scores instead of binary predictions.
# IsolationForest returns lower scores for anomalies.
# We invert the sign (-) so that higher scores represent a higher probability of being an anomaly (fire).
anomaly_scores_valid = -model.decision_function(valid_pixels)

# Reconstruct the 2D spatial map with the computed anomaly scores
# Initialize an array with NaNs, then fill only the valid pixel locations
anomaly_map = np.full((height, width), np.nan, dtype=np.float32)
anomaly_map[valid_mask] = anomaly_scores_valid


# 5. VISUALIZATION AND EXPORT

In [ ]:

# Plot the continuous anomaly scores
plt.figure(figsize=(10, 10))
# Using vmin and vmax to focus on the standard range of decision function scores
plt.imshow(anomaly_map, cmap='coolwarm', vmin=-0.2, vmax=0.2)
plt.title('Continuous Anomaly Scores (Burned Area Probability)', fontsize=15)
plt.colorbar(label='Anomaly Score (Higher = More Anomalous)', shrink=0.8)
plt.tight_layout()
plt.show()

# Update the raster profile to accommodate float32 data and explicitly define NoData
export_profile = ref_profile.copy()
export_profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan
)

# Export the results as a new GeoTIFF
output_file_path = r"data\anomaly_scores_combined.tif"
with rasterio.open(output_file_path, "w", **export_profile) as dst:
    dst.write(anomaly_map, 1)

print(f"Continuous anomaly scores successfully saved to: {output_file_path}")

# 6. MODEL VALIDATION (ROC & AUC)

In [ ]:
# Load the ground truth mask (binary raster: 1 = fire, 0 = no fire)
ground_truth_path = r"data\raster_incendio_new.tif"
with rasterio.open(ground_truth_path) as gt_src:
    # Ensure the ground truth is read at the same resolution/extent
    # If it is not perfectly aligned, you must pass it through reproject_raster first.
    ground_truth = gt_src.read(1)

# Extract ground truth values exclusively where our satellite data is valid
gt_valid = ground_truth[valid_mask]

# Compute the ROC curve using the continuous scores
fpr, tpr, thresholds = roc_curve(gt_valid, anomaly_scores_valid)

# Calculate the Area Under the Curve (AUC)
roc_auc = auc(fpr, tpr)

print(f"Model AUC: {roc_auc:.4f}")

# Plot the ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=2) # Random chance diagonal
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) for Burned Area Detection')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()